# FinalGeo interim checkpoint inspection — validation only

Reads completed seed-3/4 checkpoints and evaluates only the fixed CIFAR-100 validation split. The 10k test split remains sealed, so the final confirmatory protocol is not invalidated.

## Checkout
Create Kaggle secret `github_token`, enable Internet, and select T4 x2. Attach either a direct FinalGeo run output or one `finalgeo-resume*.zip`.

In [ ]:
import os, subprocess, sys, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN_RUNTIME': github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Expected T4x2, got {torch.cuda.device_count()} GPU(s)'
print([torch.cuda.get_device_name(i) for i in range(2)])

## Locate completed checkpoints

In [ ]:
INPUT_ROOT = Path('/kaggle/input')  # Narrow this if multiple FinalGeo bundles are attached.
def is_run_root(root):
    return ((root/'resolved_config.yaml').is_file() and (root/'protocol'/'fixed_random_projection.pt').is_file() and any((root/m/'seed_3'/'checkpoint.pt').is_file() for m in ['uniform','puregeo','finalgeo']))
roots = sorted({p.parent for p in INPUT_ROOT.rglob('resolved_config.yaml') if is_run_root(p.parent)})
if len(roots) == 1:
    RUN_ROOT = roots[0]
elif not roots:
    archives = sorted([*INPUT_ROOT.rglob('finalgeo-resume*.zip'), *INPUT_ROOT.rglob('finalgeo-confirmatory-results*.zip')])
    assert len(archives) == 1, f'Expected one FinalGeo checkpoint archive, found: {archives}'
    destination = Path('/kaggle/working/materialized-finalgeo-checkpoints'); destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as bundle:
        base = destination.resolve()
        for member in bundle.infolist():
            target = (destination/member.filename).resolve()
            assert target == base or base in target.parents, f'Unsafe ZIP member: {member.filename}'
        bundle.extractall(destination)
    roots = sorted({p.parent for p in destination.rglob('resolved_config.yaml') if is_run_root(p.parent)})
    assert len(roots) == 1, f'Archive does not contain one run root: {roots}'
    RUN_ROOT = roots[0]
else:
    raise RuntimeError(f'Multiple FinalGeo run roots found: {roots}')
print('Checkpoint root:', RUN_ROOT)
for seed in [3,4]:
    print('seed', seed, {m:(RUN_ROOT/m/f'seed_{seed}'/'checkpoint.pt').is_file() for m in ['uniform','puregeo','finalgeo']})

## Dense validation and geometry
Only seeds with all three completed checkpoints are evaluated.

In [ ]:
import json
from scripts.run_finalgeo_interim_validation import run_interim_validation
OUTPUT_ROOT = Path('/kaggle/working/finalgeo-interim-validation')
status = run_interim_validation(RUN_ROOT, OUTPUT_ROOT, seeds=[3,4], gpu_ids=[0,1])
print(json.dumps(status, indent=2))

In [ ]:
import pandas as pd
from IPython.display import display
summary = pd.read_csv(OUTPUT_ROOT/'interim_validation_method_summary.csv')
display(summary.sort_values(['seed','method']))
for seed, group in summary.groupby('seed'):
    values = group.set_index('method')
    print(f"seed {seed}: FinalGeo-Uniform common holdout = {values.loc['finalgeo','mean_common_holdout_accuracy']-values.loc['uniform','mean_common_holdout_accuracy']:+.4f}; high region = {values.loc['finalgeo','mean_high_region_accuracy']-values.loc['uniform','mean_high_region_accuracy']:+.4f}")
print('INTERIM ONLY — do not apply final confirmatory gates to these validation numbers.')

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/finalgeo-interim-validation', 'zip', root_dir=OUTPUT_ROOT)
print('Download/persist:', archive)